# Try ReFactX

In order to avoid to ingest the full 800-million-facts tree, this notebook uses a small in-memory prefix tree of 31,584 facts about famous artists and directors.

In [4]:
%load_ext autoreload
%autoreload 2

In [10]:
import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from transformers import AutoProcessor, AutoModelForImageTextToText

import refactx

In [7]:
MODEL = 'Qwen/Qwen3.5-4B'
IS_VLM = True
INDEX = '../indexes/simple_index.txt.gz'

In [8]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

In [11]:
if IS_VLM:
    processor = AutoProcessor.from_pretrained(MODEL)
    model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto')
    tokenizer = processor
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto')

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [00:02<00:00, 255.03it/s]


In [13]:
index = refactx.load_index(INDEX, tokenizer=tokenizer) # use tokenizer(text="")

100%|██████████| 316/316 [00:01<00:00, 268.62it/s]


In [14]:
streamer = TextStreamer(tokenizer)

In [15]:
question = 'Is Johnny Depp older than Brad Pitt?'

prompted_texts = [refactx.apply_prompt_template(tokenizer, question=question)]

In [16]:
#print(prompted_texts[0])

In [17]:
inputs = tokenizer(text=prompted_texts, return_tensors='pt', padding=True, padding_side='right')
inputs = inputs.to(model.device)
print(inputs['input_ids'].shape)

torch.Size([1, 778])


In [18]:
model.device

device(type='cuda', index=0)

In [19]:
# no need for num_beams=1
refactx.patch_model(model)

In [20]:
num_beams = 1
num_batches = 1

auto_streamer = streamer if num_beams == 1 else None

In [21]:
constrained_processor = refactx.get_constrained_logits_processor(tokenizer, index, num_beams, num_batches)

In [22]:
logits_processor_list = constrained_processor

model.eval()
start = time.time()

with torch.no_grad():
    out = model.generate(
        **inputs,
        logits_processor=logits_processor_list,
        max_new_tokens=800,
        streamer = auto_streamer,
        do_sample = False,
        temperature = None,
        top_k=None,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        use_cache=True,
        top_p=None,
        min_p=None,
    )

print('Elapsed', time.time() - start)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


<|im_start|>system
You are a helpful question-answering assistant that bases its answers on facts from a knowledge base and always respects the prompt.

The process to answer questions:

    You receive an input question.

    You determine the reasoning path needed to answer the question based on the information available.

    You determine the kind of answer you are asked. It can be a yes/no, a single entity, or a list of entities. Pay attention to the questions whose answer is a list of entities (e.g. Which countries share a border with Spain?): you need to find all the answer entities and include them all in the final answer.

    You get relevant facts with the "Fact:" command. You can rely on these facts and use them a proof for your answer.
    While getting facts you continue the reasoning explaining it step by step.

    Often description or short description may be useful for answering questions.

    You conclude with a concise answer that depending on the question can be a

### Visualize ReFactX output

In [23]:
_from = len(inputs.input_ids[0]) # 0
for i in range(out.shape[0]):
    print('-'*30, sum(out[i][_from:]), len(out[i][_from:]))
    print(tokenizer.decode(out[i][_from:]))

------------------------------ tensor(5111262, device='cuda:0') 800
Thinking Process:

1.  **Analyze the Request:** The user is asking a comparative question about the ages of two people: Johnny Depp and Brad Pitt. The question is "Is Johnny Depp older than Brad Pitt?". This requires a yes/no answer based on factual evidence.

2.  **Determine Reasoning Path:**
    *   Find the birth date of Johnny Depp.
    *   Find the birth date of Brad Pitt.
    *   Compare the birth dates to determine who is older.
    *   Formulate a yes/no answer based on the comparison.

3.  **Retrieve Facts:**
    *   I need to use the "Fact:" command to get information about Johnny Depp's birth date.
    *   I need to use the "Fact:" command to get information about Brad Pitt's birth date.

4.  **Execute Fact Retrieval:**
    *   Fact 1: Search for Johnny Depp's birth date.
    *   Fact 2: Search for Brad Pitt's birth date.

5.  **Compare and Conclude:**
    *   Compare the dates.
    *   Determine if Johnny D

### Generated Facts

In [24]:
for i, triple in enumerate(refactx.get_constrained_states()[0][0].generated_triples):
    print(i, tokenizer.decode(triple), end='\n')

0  <Johnny Depp> <date of birth> <1963-06-09T00:00:00Z> .
1  <Brad Pitt> <date of birth> <1963-12-18T00:00:00Z> .
2  <Sandra Bullock> <description> <American-German actress and producer> .
